# 02 Karnataka rainfall: gauge vs grid

Question (T107 only): how do the Karnataka rain-gauge totals compare with the gridded analysis,
where do the gauges cover, and what is the bias? A **seasonal** comparison is impossible: the
only gauge feed, S29 (KSNDMC via OpenCity, open licence), is annual only (D113). This notebook
compares **annual district totals** for 2020, 2021, 2022 and 2024 (the 2023 file holds 5,266
station totals only, no district rows and no coordinates), and reports station coverage for 2023.

Labels. S29 values are totals published by KSNDMC; we change nothing. S08 (IMD 0.25 degree daily
rain, local only: IMD forbids reproduction, so only derived statistics are exported) is a
**reconstruction**: a gridded analysis interpolated from gauges, area-weighted to the S17
district polygons (gappy cells dropped, D40). Nothing is imputed by us: a district-year with no
like-for-like pair is reported as excluded or missing and never filled. S29 district names reach
S17 only through the committed crosswalk `configs/crosswalks/s29_s17_districts.csv` (D119). S29
"Ballari" from 2021 excludes Vijayanagara, which S17 "Bellary" includes, and Vijayanagara has no
S17 polygon: both are excluded for 2021 on. Bias is grid minus gauge. The tracked outputs
carry only ratio statistics over several years per district, shares and counts (D166, D169); the two charts show mm and stay
local.

Run from the repository root in a fresh Python 3.12 process. Outputs go to
`reports/eda/02_ka_rain/` (or `ATLAS_EDA_OUT`).

In [ ]:
import csv
import json
import os
import statistics
import textwrap
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from atlas.analysis.quality import caption
from atlas.analysis.rain import annual_grid_totals, compare, load_crosswalk, station_coverage
from atlas.config import load_state
from atlas.contracts import RawArtifact
from atlas.features.m05 import district_monthly_rain, year_artifacts
from atlas.ingest.raw_store import RawStore
from atlas.ingest.s17 import S17
from atlas.ingest.s29 import FILES, S29

OUT = Path(os.environ.get("ATLAS_EDA_OUT", "reports/eda/02_ka_rain"))
OUT.mkdir(parents=True, exist_ok=True)
RAW = Path("data/raw")
YEARS = [2020, 2021, 2022, 2024]  # S29 years with district rows
CROSSWALK = load_crosswalk(Path("configs/crosswalks/s29_s17_districts.csv"))
NOT_LIKE_FOR_LIKE = {  # from 2021: Vijayanagara was carved out of Ballari
    "BALLARI": "S29 excludes Vijayanagara from 2021, S17 Bellary includes it",
    "VIJAYANAGAR": "no S17 polygon",
}
EXCLUDE = {(n, y): why for n, why in NOT_LIKE_FOR_LIKE.items() for y in YEARS if y >= 2021}
ka = load_state(Path("configs/states/ka.yaml"))


def write_csv(name, rows):
    with open(OUT / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]), lineterminator="\n")
        w.writeheader()
        w.writerows(rows)


def s29(year):
    entry = RawStore(RAW).latest_ok(FILES[year])
    a = entry["artifact"]
    raw = RawArtifact.model_validate(a | {"path": Path(a["path"])})
    return S29("ka").parse(raw, ka.source("S29"), year=year).to_pylist()


def s17(eid):
    manifest = json.loads(Path("tests/fixtures/s17/manifest.json").read_text("utf-8"))
    e = next(x for x in manifest["evidence"] if x["id"] == eid)
    return RawArtifact(
        sha256=e["sha256"],
        path=RAW / e["raw_path"],
        fetched_at=e["fetched_at"],
        media_type=e["media_type"],
        response_status=e["status"],
        source_url=e["url"],
        schema_mapping_version="s17-v1",
    )

In [ ]:
gauge_all = {y: s29(y) for y in [*YEARS, 2023]}
gauge = [r for y in YEARS for r in gauge_all[y] if r["area_level"] == "district"]
stations = [r for r in gauge_all[2023] if r["area_level"] == "station"]
districts = S17("ka").parse(s17("e06"), ka.source("S17"), adm1=s17("e05"))
raws = year_artifacts(RAW, range(min(YEARS), max(YEARS) + 1))
monthly = district_monthly_rain(raws, districts).to_pylist()
grid = annual_grid_totals(monthly)
print(len(gauge), "S29 district-years;", len(stations), "S29 stations (2023);", len(grid), "grid")

In [ ]:
rows = compare(gauge, grid, CROSSWALK, EXCLUDE)
S08_ROW = {"grid_mm", "bias_mm", "ratio"}  # D166, D169: no S08 value per district-year
write_csv(
    "district_year_comparison.csv",
    [{k: v for k, v in r.items() if k not in S08_ROW} for r in rows],
)
per_district = {}
for r in rows:
    if r["status"] == "compared":
        per_district.setdefault(r["district"], []).append(r["ratio"])
write_csv(  # one row over several years cannot be inverted; a single year would be one
    "district_ratio_summary.csv",
    [
        {
            "district": d,
            "years_compared": len(v),
            "grid_wetter_years": sum(x > 1.10 for x in v),
            "grid_drier_years": sum(x < 0.90 for x in v),
            "mean_ratio": statistics.fmean(v),
        }
        for d, v in per_district.items()
        if len(v) > 1
    ],
)
ok = [r for r in rows if r["status"] == "compared"]
excluded = [r for r in rows if r["status"].startswith("excluded")]
missing = [r for r in rows if r["status"] == "missing"]
biases = [r["bias_mm"] for r in ok]
ratios = [r["ratio"] for r in ok]
print(len(ok), "compared;", len(excluded), "excluded;", len(missing), "missing of", len(rows))
print("median bias", statistics.median(biases), "mm; mean", statistics.fmean(biases), "mm")
coverage = station_coverage(stations)
write_csv("station_coverage_2023.csv", coverage)
print(len(coverage), "districts;", sum(c["with_value"] for c in coverage), "of", len(stations))

In [ ]:
cap = caption(
    "S29 KSNDMC district totals vs S08 IMD 0.25 deg grid (reconstruction)",
    "annual rain, bias = grid - gauge",
    "mm",
    f"{min(YEARS)}-01-01",
    f"{max(YEARS)}-12-31 (years 2020, 2021, 2022, 2024)",
    f"{len(ok)} of {len(rows)} district-years compared",
)

cap_pub = caption(  # the summary carries the unitless ratio, not mm
    "S29 KSNDMC district totals vs S08 IMD 0.25 deg grid (reconstruction)",
    "annual rain, index = grid / gauge",
    "ratio",
    f"{min(YEARS)}-01-01",
    f"{max(YEARS)}-12-31 (years 2020, 2021, 2022, 2024)",
    f"{len(ok)} of {len(rows)} district-years compared",
)

fig, ax = plt.subplots(figsize=(6, 6))
fig.subplots_adjust(bottom=0.12)
ax.scatter([r["gauge_mm"] for r in ok], [r["grid_mm"] for r in ok], s=14)
top = max(max(r["gauge_mm"], r["grid_mm"]) for r in ok) * 1.05
ax.plot([0, top], [0, top], color="k", lw=0.8)
ax.set_xlabel("S29 gauge annual rain (mm)")
ax.set_ylabel("S08 grid annual rain (mm)")
ax.set_title("Annual district rain: grid vs gauge (1:1 line)", fontsize=10)
fig.text(0.01, 0.005, textwrap.fill(cap, 100), fontsize=5)
fig.savefig(OUT / "gauge_vs_grid.png", dpi=80, metadata={"Software": None})
plt.close(fig)

names = sorted(CROSSWALK)
by = {(r["district"], r["year"]): r["bias_mm"] for r in ok}
cells = [[by.get((n, y), np.nan) for y in YEARS] for n in names]
lim = np.nanmax(np.abs(cells))
fig, ax = plt.subplots(figsize=(6, 9))
fig.subplots_adjust(left=0.3, bottom=0.08)
im = ax.imshow(cells, aspect="auto", cmap="RdBu", vmin=-lim, vmax=lim)
ax.set_xticks(range(len(YEARS)), [str(y) for y in YEARS])
ax.set_yticks(range(len(names)), names, fontsize=7)
fig.colorbar(im, label="Bias, grid - gauge (mm; blank = excluded)")
ax.set_title("Annual bias by district", fontsize=10)
fig.text(0.01, 0.005, textwrap.fill(cap, 100), fontsize=5)
fig.savefig(OUT / "bias_by_district.png", dpi=80, metadata={"Software": None})
plt.close(fig)

In [ ]:
gauge_fetch = sorted(r["fetched_at"] for y in [*YEARS, 2023] for r in gauge_all[y][:1])
grid_fetch = sorted(r.fetched_at for r in raws)
summary = {
    "s29": {
        "district_years": len(gauge),
        "compared": len(ok),
        "excluded": len(excluded),
        "missing": len(missing),
        "excluded_by_district": {
            n: sum(r["district"] == n for r in excluded) for n in NOT_LIKE_FOR_LIKE
        },
        "years": YEARS,
        "station_year": 2023,
        "stations": len(stations),
        "stations_with_value": sum(c["with_value"] for c in coverage),
        "station_districts": len(coverage),
        "first_fetch": gauge_fetch[0].isoformat(),
        "last_fetch": gauge_fetch[-1].isoformat(),
    },
    "s08": {
        "year_files": len(raws),
        "data_start": f"{min(YEARS)}-01-01",
        "data_end": f"{max(YEARS)}-12-31",
        "first_fetch": grid_fetch[0].isoformat(),
        "last_fetch": grid_fetch[-1].isoformat(),
        "grid_district_years": len(grid),
        "grid_district_years_null": sum(v is None for v in grid.values()),
    },
    "ratio_grid_to_gauge": {
        "median": statistics.median(ratios),
        "mean": statistics.fmean(ratios),
        "min": min(ratios),
        "max": max(ratios),
        "grid_wetter_share": sum(r > 1 for r in ratios) / len(ratios),
    },
    "caption": cap_pub,
    "limits": [
        "Seasonal comparison impossible: S29 is annual only (D113); 2025 file refused by adapter.",
        "2023 has station totals only (no district rows, no coordinates): no gauge-grid pairs.",
        "S29 'year' is taken as the calendar year (adapter valid_at 31 Dec).",
        "Ballari 2021+ and Vijayanagara 2021+ are not like-for-like with S17 and are excluded.",
        "S08 is a gauge-based analysis (reconstruction), so it is not independent of the gauges.",
        "S08 is local only (IMD): only ratios, shares and counts are exported, no S08 rain in "
        "mm (D166); the two charts show mm and are not tracked.",
    ],
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", "utf-8", newline="\n")
print(json.dumps(summary, indent=2))